<a href="https://colab.research.google.com/github/mjmousavi97/C-Tehran-uni/blob/main/projects/pro-001/src/5_flowers_dataset_with_1_hidden_layer_with_reg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras


import wandb
wandb.login()
from wandb.integration.keras import WandbMetricsLogger

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohammadjavad-mousavi97 (mohammadjavad-mousavi97-arak-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [15]:
sweep_config = {
    'method': 'grid',
    'metric': {
        'name': 'val_accuracy',
        'goal': 'maximize'

        },
    'parameters': {
        'batch_size': {'values': [8]},
        'learning_rate': {'values': [ 0.0001]},
        'hidden_nodes': {'values': [128]},
        'img_size': {'values': [16]},
        'optimizer': {'values': ['adam']},
        'epochs': {'values': [10]}
        },
    }
sweep_id = wandb.sweep(
    sweep_config,
    project="5_Flower_Classification_With_HiddenLayer_with_regularization"
)

Create sweep with ID: hmh4x30f
Sweep URL: https://wandb.ai/mohammadjavad-mousavi97-arak-university/5_Flower_Classification_With_HiddenLayer_with_regularization/sweeps/hmh4x30f


In [16]:
def train():
    with wandb.init() as run:
        config = wandb.config

        IMG_HEIGHT = config.img_size
        IMG_WIDTH = config.img_size
        IMG_CHANNELS = 3

        Class_names = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]

        def read_and_decode(filename, resize_dims):
            # Read the raw file
            img_bytes = tf.io.read_file(filename=filename)
            # Decode image data
            img = tf.image.decode_jpeg(img_bytes, channels=IMG_CHANNELS)
            # Convert pixel value to float in [0, 1]
            img = tf.image.convert_image_dtype(img, tf.float32)
            # Resize the image
            img = tf.image.resize(img, resize_dims)

            return img

        def parse_csvline(csv_line):
            # record_default specify the data types for each columns
            record_default = ["", ""]
            filename, label_string = tf.io.decode_csv(csv_line, record_default)
            # load the image
            img = read_and_decode(filename, [IMG_HEIGHT, IMG_WIDTH])
            # convert label string to integer
            label = tf.argmax(tf.math.equal(Class_names, label_string))

            return img, label

        train_dataset = (
            tf.data.TextLineDataset("gs://cloud-ml-data/img/flower_photos/train_set.csv")
            .map(parse_csvline)
            .batch(config.batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

        eval_dataset = (
            tf.data.TextLineDataset("gs://cloud-ml-data/img/flower_photos/eval_set.csv")
            .map(parse_csvline)
            .batch(config.batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

        regularizer = tf.keras.regularizers.L2(0.0010)

        model = keras.Sequential([
            keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)),
            keras.layers.Flatten(),
            keras.layers.Dense(config.hidden_nodes, kernel_regularizer=regularizer, activation='relu'),
            keras.layers.Dense(len(Class_names), kernel_regularizer=regularizer, activation='softmax')]
        )

        if config.optimizer == 'adam':
            optimizer=keras.optimizers.Adam(learning_rate=config.learning_rate)
        else:
            optimizer=keras.optimizers.SGD(learning_rate=config.learning_rate)

        model.compile(
            optimizer=optimizer,
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
            metrics=["accuracy"]
        )

        history = model.fit(
            train_dataset,
            validation_data= eval_dataset,
            epochs=config.epochs,
            callbacks=[WandbMetricsLogger(log_freq=5)]
        )

In [17]:
wandb.agent(sweep_id=sweep_id, function=train)

wandb: Agent Starting Run: gblpt6hw with config:
wandb: 	batch_size: 8
wandb: 	epochs: 10
wandb: 	hidden_nodes: 128
wandb: 	img_size: 16
wandb: 	learning_rate: 0.0001
wandb: 	optimizer: adam
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/10


Traceback (most recent call last):
  File "/tmp/ipython-input-265345783.py", line 68, in train
    history = model.fit(
              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 296, in _run_job

  File "/tmp/ipython-input-265345783.py", line 68, in train

  File "/usr/local/lib/python3.12/d

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", line 296, in _run_job
    self._function()
  File "/tmp/ipython-input-265345783.py", line 68, in train
    history = model.fit(
              ^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/eager/execute.py", line 59, in quick_execute
    except TypeError as e:
tensorflow.python.framework.errors_impl.PermissionDeniedError: Graph execution error:

Detected at node IteratorGetNext defined at (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap

  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner

  File "/usr/lib/python3.12/threading.py", line 1012, in run

  File "/usr/local/lib/python3.12/dist-packages/wandb/agents/pyagent.py", li